In [41]:
!ls /content/ASL_HG

ASL_HG_36000  data


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tensorflow opencv-python matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!unzip -q /content/ASL_HG/ASL_Processed_Images.zip -d /content/ASL_HG/train

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [25]:
ls "/content/ASL_HG/ASL-HG American Sign Language Hand Gesture Image D"


ASL_HG_36000/


In [31]:
!mv "/content/ASL_HG/ASL-HG American Sign Language Hand Gesture Image D/ASL_HG_36000" \
   /content/ASL_HG/


In [34]:
!ls /content/ASL_HG/

ASL_HG_36000


In [33]:
rm -rf "/content/ASL_HG/ASL-HG American Sign Language Hand Gesture Image D"


In [36]:
!unzip -q /content/ASL_HG/ASL_HG_36000/ASL_Processed_Images.zip \
      -d /content/ASL_HG/data


In [39]:
!ls /content/ASL_HG/data/asl_processed/test

0  2  4  6  8  A  C  E	G  I  K  M  O  Q  S  U	W  Y
1  3  5  7  9  B  D  F	H  J  L  N  P  R  T  V	X  Z


In [40]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ------------------
# CONFIG
# ------------------
IMG_SIZE = 224
BATCH = 64
EPOCHS = 20

TRAIN_DIR = "/content/ASL_HG/data/asl_processed/train"
TEST_DIR  = "/content/ASL_HG/data/asl_processed/test"

# ------------------
# DATA GENERATORS
# ------------------
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
print("Classes:", train_data.class_indices)

# ------------------
# MODEL (MobileNetV2)
# ------------------
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # phase 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ------------------
# CALLBACKS
# ------------------
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

# ------------------
# TRAIN
# ------------------
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15,
    callbacks=callbacks
)

# ------------------
# SAVE MODEL
# ------------------
model.save("/content/asl_hg_mobilenet.keras")
model.export("/content/asl_hg_savedmodel")

print("✅ Training complete and model saved")


Found 28800 images belonging to 36 classes.
Found 7200 images belonging to 36 classes.
Classes: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'A': 10, 'B': 11, 'C': 12, 'D': 13, 'E': 14, 'F': 15, 'G': 16, 'H': 17, 'I': 18, 'J': 19, 'K': 20, 'L': 21, 'M': 22, 'N': 23, 'O': 24, 'P': 25, 'Q': 26, 'R': 27, 'S': 28, 'T': 29, 'U': 30, 'V': 31, 'W': 32, 'X': 33, 'Y': 34, 'Z': 35}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,595,172 (9.90 MB)

 Trainable params: 337,188 (1.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 400s 847ms/step - accuracy: 0.4940 - loss: 1.7904 - val_accuracy: 0.9810 - val_loss: 0.1337 - learning_rate: 0.0010
Epoch 2/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 364s 808ms/step - accuracy: 0.8892 - loss: 0.3496 - val_accuracy: 0.9918 - val_loss: 0.0533 - learning_rate: 0.0010
Epoch 3/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 364s 808ms/step - accuracy: 0.9414 - loss: 0.1981 - val_accuracy: 0.9996 - val_loss: 0.0233 - learning_rate: 0.0010
Epoch 4/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 364s 810ms/step - accuracy: 0.9532 - loss: 0.1516 - val_accuracy: 0.9997 - val_loss: 0.0107 - learning_rate: 0.0010
Epoch 5/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 366s 814ms/step - accuracy: 0.9645 - loss: 0.1163 - val_accuracy: 0.9987 - val_loss: 0.0122 - learning_rate: 0.0010
Epoch 6/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 372s 828ms/step - accuracy: 0.9663 - loss: 0.1043 - val_accuracy: 0.9994 - val_loss: 0.0107 - learning_rate: 0.0010
Epoch 7/20
450/450 ━━━━━━━━━━━━━━━━━━━━ 370s 822ms/step - accura

In [42]:
# ------------------
# SAVE MODEL
# ------------------
model.save("/content/asl_hg_mobilenet.keras")
model.export("/content/asl_hg_savedmodel")

print("✅ Training complete and model saved")

# download keras model
from google.colab import files
files.download("/content/asl_hg_mobilenet.keras")

# zip + download SavedModel
!zip -r asl_hg_savedmodel.zip /content/asl_hg_savedmodel
files.download("asl_hg_savedmodel.zip")


Saved artifact at '/content/asl_hg_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 36), dtype=tf.float32, name=None)
Captures:
  134562564066768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564068496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564070800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564069648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564069840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564068304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: content/asl_hg_savedmodel/ (stored 0%)
  adding: content/asl_hg_savedmodel/assets/ (stored 0%)
  adding: content/asl_hg_savedmodel/variables/ (stored 0%)
  adding: content/asl_hg_savedmodel/variables/variables.data-00000-of-00001 (deflated 7%)
  adding: content/asl_hg_savedmodel/variables/variables.index (deflated 78%)
  adding: content/asl_hg_savedmodel/fingerprint.pb (stored 0%)
  adding: content/asl_hg_savedmodel/saved_model.pb (deflated 90%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
# ------------------
# SAVE MODEL
# ------------------
model.save("/content/asl_hg_mobilenet.keras")   # Keras format
model.save("/content/asl_hg_mobilenet.h5")      # H5 format
model.export("/content/asl_hg_savedmodel")      # TensorFlow SavedModel

print("✅ Training complete and model saved")


Saved artifact at '/content/asl_hg_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 36), dtype=tf.float32, name=None)
Captures:
  134562564066768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564068496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564070800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564069648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564069840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564071568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562564068304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134

In [44]:
from google.colab import files
files.download("/content/asl_hg_mobilenet.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>